In [2]:
# =====================================================================
# LAB 3: CHATBOT DE E-COMMERCE COM DECISION TREE
# =====================================================================
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
dados_treino = [
    ("Onde esta meu pedido e qual o rastreio", "rastrear_pedido"),
    ("Quero saber a localizacao da minha encomenda", "rastrear_pedido"),
    ("Quero cancelar minha compra e pedir reembolso", "cancelar_compra"),
    ("Preciso cancelar o pedido feito ontem", "cancelar_compra"),
    ("Qual o horario de atendimento da loja fisica", "fora_escopo")
]
stopwords = ["o", "a", "meu", "minha", "e", "da", "do"]
# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 3.1: Filtre as stopwords da frase
    tokens_filtrados = [t for t in tokens if t not in stopwords]

    return " ".join(tokens_filtrados)
X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]
# 2. VETORIZAÇÃO E TREINAMENTO (DECISION TREE)
vectorizer = TfidfVectorizer()
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)
# TODO 3.2: Instancie o modelo DecisionTreeClassifier
modelo_tree = DecisionTreeClassifier()
# TODO 3.3: Treine o modelo com os dados vetorizados e os rótulos de treino
modelo_tree.fit(X_vetorizado, y_treino)

DecisionTreeClassifier()

In [3]:
# 3. EXTRAÇÃO DE ENTIDADE (CÓDIGO DE RASTREIO BR)
def extrair_codigo_rastreio(texto):
    match = re.search(r'BR\d{9}', texto, re.IGNORECASE)
    # TODO 3.4: Se encontrar, retorne o texto em maiúsculas. Se não, None.
    return match.group(0).upper() if match else None
# 4. PIPELINE NLU
def processar_ecommerce(mensagem_usuario, threshold=0.50):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo_tree.predict_proba(vetor_input)[0]

    maior_confianca = max(probas)
    intencao_prevista = modelo_tree.classes_[probas.argmax()]

    # TODO 3.5: Verificação de segurança
    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {"status": "FALLBACK", "confianca": round(maior_confianca, 4), "mensagem": "Nao entendi. Vou transferir para um atendente."}
    else:
        codigo = extrair_codigo_rastreio(mensagem_usuario)
        return {"status": "SUCESSO", "intencao": intencao_prevista, "confianca": round(maior_confianca, 4), "codigo_rastreio": codigo}


In [4]:
# --- TESTE 3 ---
if __name__ == "__main__":
    print("=== LAB 3 - DECISION TREE E-COMMERCE ===")
    print()
    print("Teste 1:", processar_ecommerce("Quero rastrear o pedido BR987654321"))
    print("Teste 2:", processar_ecommerce("Preciso cancelar minha compra"))
    print("Teste 3:", processar_ecommerce("Qual o horario de atendimento da loja fisica"))
    print("Teste 4:", processar_ecommerce("Onde esta minha encomenda BR123456789"))

=== LAB 3 - DECISION TREE E-COMMERCE ===

Teste 1: {'status': 'SUCESSO', 'intencao': np.str_('rastrear_pedido'), 'confianca': np.float64(1.0), 'codigo_rastreio': 'BR987654321'}
Teste 2: {'status': 'SUCESSO', 'intencao': np.str_('cancelar_compra'), 'confianca': np.float64(1.0), 'codigo_rastreio': None}
Teste 3: {'status': 'FALLBACK', 'confianca': np.float64(1.0), 'mensagem': 'Nao entendi. Vou transferir para um atendente.'}
Teste 4: {'status': 'SUCESSO', 'intencao': np.str_('rastrear_pedido'), 'confianca': np.float64(1.0), 'codigo_rastreio': 'BR123456789'}
